In [1]:
import pandas as pd
import numpy as np
import math
from collections import Counter

In [ ]:
def Entropy(input):
    if not input:
        return 0.0
    list_size = len(input)
    counts = Counter(input)
    entropy = 0.0
    for count in counts.values():
        p = count/list_size
        if p > 0:
            entropy -= p * math.log2(p)
    return entropy

def IG(parent,left,right):
    p_entropy = Entropy(parent)
    p_size = len(parent)
    
    w_left = len(left) / p_size
    w_right = len(right) / p_size
    
    c_entropy = (w_left * Entropy(left)) + (w_right * Entropy(right))
    return p_entropy - c_entropy
    

In [3]:
df = pd.read_csv("drug200.csv")
df.head()
df.info()

train = df.sample(frac=0.66,random_state=222)
test = df.drop(train.index)

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          200 non-null    int64  
 1   Sex          200 non-null    str    
 2   BP           200 non-null    str    
 3   Cholesterol  200 non-null    str    
 4   Na_to_K      200 non-null    float64
 5   Drug         200 non-null    str    
dtypes: float64(1), int64(1), str(4)
memory usage: 9.5 KB


In [ ]:
def find_best_split(data,t_col='Drug'):
    
    best_gain = -1.0
    best_feature = None
    best_threshold = None
    best_is_cat = False
    
    parent_y = data[t_col].tolist()
    feature_cols = [c for c in data.columns if c != t_col]
    
    for col in feature_cols:
        col_data = data[col]
        is_cat = col_data.dtype == 'object'
        
        if is_cat:
            for val in col_data.unique():
                left_mask = data[col] == val
                left_y = data.loc[left_mask,t_col].tolist()
                right_y = data.loc[~left_mask,t_col].tolist()
                
                gain = IG(parent_y, left_y, right_y)
                if gain > best_gain:
                    best_gain, best_feature, best_threshold, best_is_cat = gain, col, val, True
        else:
            unique_vals = np.sort(col_data.unique())
            if len(unique_vals) < 2:
                continue
            midpoints = [(unique_vals[i] + unique_vals[i+1]) / 2.0 for i in range(len(unique_vals) - 1)]
            
            for mid in midpoints:
                left_mask = data[col] <= mid
                left_y = data.loc[left_mask, t_col].tolist()
                right_y = data.loc[~left_mask, t_col].tolist()
                
                gain = IG(parent_y, left_y, right_y)
                if gain > best_gain:
                    best_gain, best_feature, best_threshold, best_is_cat = gain, col, mid, False
    
    print(best_feature,best_threshold,best_is_cat,best_gain)                
    return best_feature, best_threshold, best_is_cat, best_gain


feat, thresh, is_cat, gain = find_best_split(train)
print(f"Best Initial Split -> Feature: {feat} | Threshold: {thresh} | Gain: {gain:.4f}")

Na_to_K 14.838999999999999 False 0.9958560657511164
Best Initial Split -> Feature: Na_to_K | Threshold: 14.838999999999999 | Gain: 0.9959


In [37]:
def build_tree(data,target_col='Drug',depth=0,max_depth=4,min_samples_split=2):
    target_list = data[target_col].tolist()
    num_samples = len(data)
    unique_classes = set(target_list)
    
    majority_class = Counter(target_list).most_common(1)[0][0] if target_list else None
    
    if len(unique_classes) == 1 or depth >= max_depth or num_samples < min_samples_split:
        return {'is_leaf': True, 'value': majority_class}
    
    feature, threshold, is_cat, gain = find_best_split(data,target_col)

    if gain <= 0 or feature is None:
        return {'is_leaf': True, 'value': majority_class}
    
    if is_cat:
        left_df = data[data[feature] == threshold]
        right_df = data[data[feature] != threshold]
    else:
        left_df = data[data[feature] <= threshold]
        right_df = data[data[feature] > threshold]
        
    return {
        'is_leaf': False,
        'feature': feature,
        'threshold': threshold,
        'is_categorical': is_cat,
        'left': build_tree(left_df, target_col, depth + 1, max_depth, min_samples_split),
        'right': build_tree(right_df, target_col, depth + 1, max_depth, min_samples_split)
    }

tree = build_tree(train, max_depth=4, min_samples_split=2)

Na_to_K 14.838999999999999 False 0.9958560657511164
BP HIGH True 0.8930108100446013
Age 50.5 False 1.0
BP LOW True 0.27474579735739035
Cholesterol NORMAL True 0.9957274520849255


In [39]:
def predict(node,sample_row):
    if node['is_leaf']:
        return node['value']
    val = sample_row[node['feature']]
    if node['is_categorical']:
        next_node = node['left'] if val == node['threshold'] else node['right']
    else:
        next_node = node['left'] if val <= node['threshold'] else node['right']
        
    return predict(next_node, sample_row)


predictions = [predict(tree, row) for _, row in test.iterrows()]
actual = test['Drug'].tolist()

correct = sum(p == a for p, a in zip(predictions, actual))
accuracy = (correct / len(actual)) * 100

results = pd.DataFrame({
    'Actual Label': actual,
    'Predicted Label': predictions,
    'Match': [p == a for p, a in zip(predictions, actual)]
})

print(f"Accuracy: {accuracy:.2f}%\n")
results

Accuracy: 100.00%



,Actual Label,Predicted Label,Match
0,drugY,drugY,True
1,drugY,drugY,True
2,drugC,drugC,True
3,drugY,drugY,True
4,drugX,drugX,True
...,...,...,...
63,drugA,drugA,True
64,drugY,drugY,True
65,drugB,drugB,True
66,drugA,drugA,True
